# 4. Fördelningar och normalfördelningen

I förra notebooken beskrev vi data med histogram. En **fördelning** är den bakomliggande modellen:
en beskrivning av vilka värden som är vanliga och vilka som är ovanliga. Med en fördelning kan vi svara på frågor som
*"hur ovanligt är det att vara längre än 190 cm?"* utan att behöva räkna i datan varje gång.

**Efter den här notebooken kan du**
- förklara vad en fördelning är och hur den hänger ihop med ett histogram
- beskriva normalfördelningen med medelvärde μ och standardavvikelse σ
- använda 68–95–99,7-regeln
- räkna om ett värde till ett z-värde och tolka det
- räkna ut sannolikheter med `stats.norm` (`cdf`, `sf` och `ppf`)
- känna igen en sned fördelning

**Förkunskaper:** [3. Deskriptiv statistik](../03_deskriptiv_statistik/03_deskriptiv_statistik.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

## 4.1 Från slump till mönster

Ett enskilt tärningskast går inte att förutsäga. Men *många* kast följer ett tydligt mönster: varje sida kommer upp
ungefär en sjättedel av gångerna. Det mönstret är tärningens **fördelning**.

🔮 **Gissa först:** hur många kast behövs innan staplarna ser jämnhöga ut?

In [ ]:
num_rolls = 30        # ← ändra: pröva 30, 300, 3000 och 300_000

rolls = rng.integers(1, 7, num_rolls)
relative_freq = pd.Series(rolls).value_counts(normalize=True).reindex(range(1, 7), fill_value=0)

relative_freq.plot(kind="bar", rot=0)
plt.axhline(1 / 6, color="red", linestyle="--", label="teoretisk sannolikhet 1/6")
plt.xlabel("Tärningens värde")
plt.ylabel("Relativ frekvens")
plt.title(f"{num_rolls} kast")
plt.legend()
plt.show()

🧪 **Labba:** Öka antalet kast steg för steg. De relativa frekvenserna i datan närmar sig de teoretiska sannolikheterna.

- **Histogrammet/stapeldiagrammet** visar vad som *hände* i just vår data.
- **Fördelningen** beskriver vad som händer *i längden*.

## 4.2 Normalfördelningen

För kontinuerliga variabler beskrivs fördelningen av en **täthetskurva**. Den vanligaste är **normalfördelningen**,
den symmetriska "klockkurvan". Många mätvärden blir ungefär normalfördelade: kroppslängd, mätfel, provresultat.

Kurvan bestäms helt av två tal:
- **μ (my)**, medelvärdet: var kurvans mitt ligger.
- **σ (sigma)**, standardavvikelsen: hur bred kurvan är.

In [ ]:
mu, sigma = 172, 9        # ← ändra: pröva t.ex. (180, 9), (172, 3) och (172, 20)

x = np.linspace(130, 215, 400)
plt.plot(x, stats.norm.pdf(x, 172, 9), "k--", label="μ = 172, σ = 9 (referens)")
plt.plot(x, stats.norm.pdf(x, mu, sigma), color="tab:blue", label=f"μ = {mu}, σ = {sigma}")
plt.xlabel("Längd (cm)")
plt.ylabel("Täthet")
plt.legend()
plt.show()

🧪 **Labba:** Ändra först bara `mu`, sedan bara `sigma`. Vilken parameter *flyttar* kurvan? Vilken ändrar *formen*?
Varför blir kurvan lägre när den blir bredare?

<details>
<summary>💡 <b>Svar</b></summary>

μ flyttar kurvan i sidled och σ gör den smalare eller bredare. Den totala arean under kurvan är alltid 1 (100 % av alla
värden), så en bredare kurva måste bli lägre.

</details>

### Sannolikhet är area

Höjden på kurvan är ingen sannolikhet. Det är **arean under kurvan** mellan två värden som är sannolikheten
(andelen) att hamna där. Vi kontrollerar genom att simulera 10 000 längder och lägga kurvan ovanpå histogrammet:

In [ ]:
heights = rng.normal(172, 9, 10_000)

plt.hist(heights, bins=50, density=True, alpha=0.6, label="10 000 simulerade längder")
x = np.linspace(135, 210, 300)
plt.plot(x, stats.norm.pdf(x, 172, 9), color="red", label="Normalfördelning (172, 9)")
plt.xlabel("Längd (cm)")
plt.ylabel("Täthet")
plt.legend()
plt.show()

(`density=True` skalar om histogrammet så att staplarnas sammanlagda area blir 1, precis som kurvans.)

## 4.3 68–95–99,7-regeln

I *varje* normalfördelning, oavsett μ och σ, gäller ungefär:

| Inom | ligger |
|---|---|
| μ ± 1σ | 68 % av värdena |
| μ ± 2σ | 95 % av värdena |
| μ ± 3σ | 99,7 % av värdena |

För längderna (μ = 172, σ = 9) betyder det att ungefär 95 % är mellan 154 och 190 cm. Stämmer det i vår simulerade data?

In [ ]:
for k in [1, 2, 3]:
    low, high = 172 - k * 9, 172 + k * 9
    share = ((heights >= low) & (heights <= high)).mean()
    print(f"Inom ±{k}σ  ({low}–{high} cm): {share:.1%}")

Regeln är värd att kunna utantill. Den förklarar till exempel varför talet **2** (egentligen 1,96) dyker upp överallt
när vi kommer till konfidensintervall.

✏️ **Din tur:** IQ-test är konstruerade så att resultaten är normalfördelade med μ = 100 och σ = 15.
Använd regeln (ingen kod behövs) för att svara:
1. Mellan vilka värden ligger 95 % av befolkningen?
2. Ungefär hur stor andel har över 130?

<details>
<summary>💡 <b>Svar</b></summary>

1. 100 ± 2 · 15, alltså mellan 70 och 130.
2. 5 % ligger utanför 70–130. Fördelningen är symmetrisk, så hälften av dem ligger över 130: ungefär 2,5 %.

</details>

## 4.4 z-värden: hur ovanligt är ett värde?

Är 190 cm långt? Är 130 i IQ högt? För att jämföra värden från olika skalor räknar vi om dem till **z-värden**:

$$z = \frac{x - \mu}{\sigma}$$

z-värdet anger **hur många standardavvikelser** ett värde ligger från medelvärdet. z = 0 är exakt medel, z = 2 är
ovanligt högt och z = −3 är extremt lågt.

In [ ]:
value = 190          # ← ändra
z = (value - 172) / 9
print(f"{value} cm motsvarar z = {z:.2f}")

✏️ **Din tur:** Lisa är 185 cm lång (μ = 172, σ = 9) och har 125 i IQ (μ = 100, σ = 15).
Vad sticker ut mest: hennes längd eller hennes IQ? Räkna ut båda z-värdena.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
z_height = (185 - 172) / 9
z_iq = (125 - 100) / 15
print(f"z för längd: {z_height:.2f}")
print(f"z för IQ:    {z_iq:.2f}")
# IQ-värdet ligger längre från medel räknat i standardavvikelser, så det sticker ut mest.
```

</details>

## 4.5 Räkna ut sannolikheter med `stats.norm`

68–95–99,7-regeln räcker för hela standardavvikelser. För allt annat använder vi scipy.
Tre funktioner räcker långt:

| Funktion | Svarar på | Bild |
|---|---|---|
| `stats.norm.cdf(x, μ, σ)` | Hur stor andel ligger **under** x? | arean till vänster |
| `stats.norm.sf(x, μ, σ)` | Hur stor andel ligger **över** x? | arean till höger |
| `stats.norm.ppf(q, μ, σ)` | **Omvänt:** vilket x har andelen q under sig? | från area till värde |

In [ ]:
print(f"Andel kortare än 160 cm: {stats.norm.cdf(160, 172, 9):.1%}")
print(f"Andel längre än 190 cm:  {stats.norm.sf(190, 172, 9):.1%}")
print(f"De längsta 10 % är över: {stats.norm.ppf(0.90, 172, 9):.1f} cm")

Bilden nedan visar vad `sf` räknar ut. Flytta gränsen och se hur arean ändras:

In [ ]:
limit = 190          # ← ändra

x = np.linspace(135, 210, 400)
y = stats.norm.pdf(x, 172, 9)
plt.plot(x, y, color="black")
plt.fill_between(x, y, where=x >= limit, color="red", alpha=0.4,
                 label=f"Andel över {limit} cm = {stats.norm.sf(limit, 172, 9):.1%}")
plt.xlabel("Längd (cm)")
plt.ylabel("Täthet")
plt.legend()
plt.show()

En andel **mellan** två värden får du genom att ta skillnaden mellan två `cdf`:

In [ ]:
share_between = stats.norm.cdf(180, 172, 9) - stats.norm.cdf(165, 172, 9)
print(f"Andel mellan 165 och 180 cm: {share_between:.1%}")

✏️ **Din tur:**
1. Hur stor andel är mellan 170 och 175 cm?
2. En berg-och-dalbana kräver minst 150 cm. Hur stor andel av de vuxna är för korta?
3. En dörrtillverkare vill att 99 % ska kunna gå genom dörren utan att böja sig. Hur hög måste dörren minst vara?
4. Kontrollera svar 2 mot den simulerade datan i `heights` med `(heights < 150).mean()`.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
print(f"1. Mellan 170 och 175: {stats.norm.cdf(175, 172, 9) - stats.norm.cdf(170, 172, 9):.1%}")
print(f"2. Kortare än 150:     {stats.norm.cdf(150, 172, 9):.2%}")
print(f"3. Dörrhöjd:           {stats.norm.ppf(0.99, 172, 9):.1f} cm")
print(f"4. I simuleringen:     {(heights < 150).mean():.2%}")
```

</details>

### Standardnormalfördelningen

Om man anger varken μ eller σ använder scipy **standardnormalfördelningen** med μ = 0 och σ = 1. Det är fördelningen för z-värden.
Två tal därifrån som du kommer att möta många gånger:

In [ ]:
print(f"Andel inom z = ±1,96:      {stats.norm.cdf(1.96) - stats.norm.cdf(-1.96):.3f}")
print(f"z-värde med 97,5 % under:  {stats.norm.ppf(0.975):.3f}")

Exakt 95 % av en normalfördelning ligger alltså inom ±1,96 standardavvikelser. "±2σ" i 68–95–99,7-regeln är en avrundning.

## 4.6 Alla fördelningar är inte normala

Inkomster är ett typiskt exempel på en **högersned** fördelning: de flesta ligger ganska samlade, men några få har mycket höga värden.

In [ ]:
incomes = rng.lognormal(np.log(32), 0.4, 10_000)      # tkr per månad

plt.hist(incomes, bins=60, density=True, alpha=0.6, label="Inkomster")
x = np.linspace(0, 120, 300)
plt.plot(x, stats.norm.pdf(x, incomes.mean(), incomes.std()), color="red", label="Normalfördelning med samma μ och σ")
plt.axvline(incomes.mean(), color="red", linestyle=":", label="medelvärde")
plt.axvline(np.median(incomes), color="black", linestyle="--", label="median")
plt.xlabel("Inkomst (tkr per månad)")
plt.ylabel("Täthet")
plt.legend()
plt.show()

Normalkurvan passar dåligt: den hamnar för långt till höger om toppen och saknar den långa svansen.
Medelvärdet ligger över medianen, vilket är kännetecknet för en högersned fördelning.

✏️ **Din tur:** Stämmer 68–95–99,7-regeln för `incomes`? Räkna ut andelen inom ±1, ±2 och ±3 standardavvikelser
från medelvärdet. Räkna också ut hur stor andel som ligger *över* medelvärdet + 3 standardavvikelser, och jämför med
vad en normalfördelning skulle ge (`stats.norm.sf(3)`).

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
m, s = incomes.mean(), incomes.std()
for k in [1, 2, 3]:
    share = (np.abs(incomes - m) <= k * s).mean()
    print(f"Inom ±{k}σ: {share:.1%}")
print(f"Över m + 3s:         {(incomes > m + 3 * s).mean():.2%}")
print(f"Normalfördelningen:  {stats.norm.sf(3):.2%}")
# ±2σ råkar stämma bra, men ±1σ gör det inte, och i svansen blir det helt fel:
# extremt höga värden är ungefär tio gånger vanligare än normalfördelningen förutsäger.
```

</details>

Lärdomen: **titta alltid på datan** innan du antar att den är normalfördelad. Lite längre fram
([notebook 5](../05_population_och_stickprov/05_population_och_stickprov.ipynb)) får du se något överraskande:
*medelvärden* blir ungefär normalfördelade även när datan inte är det.

## Sammanfattning

- En fördelning beskriver hur värden sprider sig i längden. Sannolikhet = **area** under täthetskurvan.
- Normalfördelningen bestäms av **μ** (läge) och **σ** (bredd).
- **68–95–99,7-regeln:** andelen inom 1, 2 och 3 standardavvikelser.
- **z = (x − μ) / σ** anger hur många standardavvikelser från medel ett värde ligger.
- `cdf` ger arean till vänster, `sf` arean till höger och `ppf` går från area till värde.
- Sneda fördelningar följer inte regeln. Titta på histogrammet först.

### Kontrollfrågor
1. Vad är `stats.norm.cdf(172, 172, 9)`? Svara utan att köra koden.
2. Två prov har medel 60 (σ = 10) respektive 70 (σ = 20). Du fick 80 på båda. På vilket prov presterade du bäst jämfört med de andra?
3. Vad är `stats.norm.cdf(x) + stats.norm.sf(x)` för vilket x som helst?

<details>
<summary>💡 <b>Svar</b></summary>

1. 0,5. Halva fördelningen ligger under medelvärdet eftersom den är symmetrisk.
2. Det första: z = (80 − 60) / 10 = 2,0 mot z = (80 − 70) / 20 = 0,5.
3. 1. Arean till vänster plus arean till höger är hela arean.

</details>

**Nästa notebook:** [5. Population och stickprov](../05_population_och_stickprov/05_population_och_stickprov.ipynb)